# Paper 1: region order of the released networks (SI Appendix, Note 7)

Evidence that the HCPd, HCPya, camCAN and HCPa networks in all_data.mat follow the AAL3 region order.

Reference: CALM, RED and ACE (processed in-house with QSIprep and AAL116, i.e. AAL90 after removing 91-116).
For each other cohort, the released node order is read under two labellings and compared with the reference
on the 82 regions that AAL90 and AAL3 share:
  AAL90 labelling : node i is AAL90 region i (what the Methods state)
  AAL3 labelling  : node i is the i-th AAL3 region among labels 1-94, skipping the empty labels 35, 36, 81, 82

In [ ]:
import json, sys
import numpy as np, pandas as pd
from scipy.optimize import quadratic_assignment
import os
from pathlib import Path
REPO = Path.cwd().resolve().parent                    # this notebook lives in paper/
sys.path.insert(0, str(REPO / "chain")); import chain_analysis as ca
DATA_DIR = Path(os.environ.get("NETLEARN_DATA", "~/netlearn-data")).expanduser()

AAL90 = """Precentral_L Precentral_R Frontal_Sup_L Frontal_Sup_R Frontal_Sup_Orb_L Frontal_Sup_Orb_R Frontal_Mid_L
Frontal_Mid_R Frontal_Mid_Orb_L Frontal_Mid_Orb_R Frontal_Inf_Oper_L Frontal_Inf_Oper_R Frontal_Inf_Tri_L Frontal_Inf_Tri_R
Frontal_Inf_Orb_L Frontal_Inf_Orb_R Rolandic_Oper_L Rolandic_Oper_R Supp_Motor_Area_L Supp_Motor_Area_R Olfactory_L
Olfactory_R Frontal_Sup_Medial_L Frontal_Sup_Medial_R Frontal_Med_Orb_L Frontal_Med_Orb_R Rectus_L Rectus_R Insula_L
Insula_R Cingulum_Ant_L Cingulum_Ant_R Cingulum_Mid_L Cingulum_Mid_R Cingulum_Post_L Cingulum_Post_R Hippocampus_L
Hippocampus_R ParaHippocampal_L ParaHippocampal_R Amygdala_L Amygdala_R Calcarine_L Calcarine_R Cuneus_L Cuneus_R
Lingual_L Lingual_R Occipital_Sup_L Occipital_Sup_R Occipital_Mid_L Occipital_Mid_R Occipital_Inf_L Occipital_Inf_R
Fusiform_L Fusiform_R Postcentral_L Postcentral_R Parietal_Sup_L Parietal_Sup_R Parietal_Inf_L Parietal_Inf_R
SupraMarginal_L SupraMarginal_R Angular_L Angular_R Precuneus_L Precuneus_R Paracentral_Lobule_L Paracentral_Lobule_R
Caudate_L Caudate_R Putamen_L Putamen_R Pallidum_L Pallidum_R Thalamus_L Thalamus_R Heschl_L Heschl_R Temporal_Sup_L
Temporal_Sup_R Temporal_Pole_Sup_L Temporal_Pole_Sup_R Temporal_Mid_L Temporal_Mid_R Temporal_Pole_Mid_L
Temporal_Pole_Mid_R Temporal_Inf_L Temporal_Inf_R""".split()
AAL3_94 = """Precentral_L Precentral_R Frontal_Sup_2_L Frontal_Sup_2_R Frontal_Mid_2_L Frontal_Mid_2_R Frontal_Inf_Oper_L
Frontal_Inf_Oper_R Frontal_Inf_Tri_L Frontal_Inf_Tri_R Frontal_Inf_Orb_2_L Frontal_Inf_Orb_2_R Rolandic_Oper_L Rolandic_Oper_R
Supp_Motor_Area_L Supp_Motor_Area_R Olfactory_L Olfactory_R Frontal_Sup_Medial_L Frontal_Sup_Medial_R Frontal_Med_Orb_L
Frontal_Med_Orb_R Rectus_L Rectus_R OFCmed_L OFCmed_R OFCant_L OFCant_R OFCpost_L OFCpost_R OFClat_L OFClat_R Insula_L
Insula_R EMPTY35 EMPTY36 Cingulate_Mid_L Cingulate_Mid_R Cingulate_Post_L Cingulate_Post_R Hippocampus_L
Hippocampus_R ParaHippocampal_L ParaHippocampal_R Amygdala_L Amygdala_R Calcarine_L Calcarine_R Cuneus_L Cuneus_R Lingual_L
Lingual_R Occipital_Sup_L Occipital_Sup_R Occipital_Mid_L Occipital_Mid_R Occipital_Inf_L Occipital_Inf_R Fusiform_L
Fusiform_R Postcentral_L Postcentral_R Parietal_Sup_L Parietal_Sup_R Parietal_Inf_L Parietal_Inf_R SupraMarginal_L
SupraMarginal_R Angular_L Angular_R Precuneus_L Precuneus_R Paracentral_Lobule_L Paracentral_Lobule_R Caudate_L Caudate_R
Putamen_L Putamen_R Pallidum_L Pallidum_R EMPTY81 EMPTY82 Heschl_L Heschl_R Temporal_Sup_L Temporal_Sup_R
Temporal_Pole_Sup_L Temporal_Pole_Sup_R Temporal_Mid_L Temporal_Mid_R Temporal_Pole_Mid_L Temporal_Pole_Mid_R
Temporal_Inf_L Temporal_Inf_R""".split()
assert len(AAL90) == 90 and len(AAL3_94) == 94
AAL3 = [n for n in AAL3_94 if not n.startswith("EMPTY")]; assert len(AAL3) == 90
canon = lambda n: n.replace("_2_", "_").replace("Cingulate", "Cingulum")
SHARED = [n for n in AAL90 if n in [canon(m) for m in AAL3]]; assert len(SHARED) == 82
I90 = np.array([AAL90.index(n) for n in SHARED]); I3 = np.array([[canon(m) for m in AAL3].index(n) for n in SHARED])
NAMES = {1: "dHCP", 2: "BCP", 3: "CALM", 4: "RED", 5: "ACE", 6: "HCPd", 7: "HCPya", 8: "camCAN", 9: "HCPa"}
OUT = str(REPO / "paper/data/p1") + "/"

sid, mats, meta = ca.load_export(str(DATA_DIR))
ds = meta.set_index("sid").loc[sid, "dataset"].values
W = mats / (mats.sum((1, 2), keepdims=True) / 2)           # share of total weight (each edge counted once)
P = (mats > 0).astype(float)                               # edge presence
iu = np.triu_indices(82, 1)
ref_sel = np.isin(ds, [3, 4, 5])
refW, refP = W[ref_sel].mean(0)[np.ix_(I90, I90)], P[ref_sel].mean(0)[np.ix_(I90, I90)]
corr = lambda A, B: float(np.corrcoef(A[iu], B[iu])[0, 1])
res = {"n": {NAMES[k]: int((ds == k).sum()) for k in NAMES}, "cohort": {}}
for k in [1, 2, 6, 7, 8, 9]:
    sel = ds == k; mW, mP = W[sel].mean(0), P[sel].mean(0)
    # per subject: which labelling fits the reference edge-presence pattern better
    better3 = 0
    for A in P[sel]:
        better3 += corr(A[np.ix_(I3, I3)], refP) > corr(A[np.ix_(I90, I90)], refP)
    res["cohort"][NAMES[k]] = dict(
        rP_90=corr(mP[np.ix_(I90, I90)], refP), rP_3=corr(mP[np.ix_(I3, I3)], refP),
        rW_90=corr(mW[np.ix_(I90, I90)], refW), rW_3=corr(mW[np.ix_(I3, I3)], refW),
        n=int(sel.sum()), n_better_AAL3=int(better3))
# blind graph matching (no atlas information): reference vs each AAL3-candidate cohort, edge presence, 90 x 90
refP90 = P[ref_sel].mean(0)
res["matching"] = {}
for k in [6, 7, 8, 9]:
    B = P[ds == k].mean(0); best = None
    for seed in range(20):
        q = quadratic_assignment(refP90, B, method="faq", options=dict(maximize=True, rng=np.random.default_rng(seed), P0="randomized"))
        if best is None or q.fun > best.fun: best = q
    inv = np.argsort(best.col_ind)          # released node j <-> AAL90 node inv[j]
    exp = np.array([AAL90.index(canon(n)) if canon(n) in AAL90 else -1 for n in AAL3])
    ok = int(sum(1 for j in range(90) if exp[j] >= 0 and inv[j] == exp[j]))
    ok90 = int(sum(1 for j in range(90) if inv[j] == j))
    res["matching"][NAMES[k]] = dict(agree_AAL3=ok, of=82, agree_AAL90=ok90)
# named connections: read at the AAL90 index and at the AAL3 index
yeh = np.isin(ds, [6, 7, 8, 9])
def share(sel, i, j): return float(W[sel][:, i, j].mean() * 1000)
CONN = [("Supp_Motor_Area_L", "Supp_Motor_Area_R"), ("Precuneus_L", "Precuneus_R"), ("Precentral_L", "Postcentral_L"),
        ("Calcarine_L", "Lingual_L"), ("Frontal_Inf_Orb_L", "Frontal_Inf_Orb_R")]
res["connections"] = []
for a, b in CONN:
    i90, j90 = AAL90.index(a), AAL90.index(b); i3, j3 = [canon(m) for m in AAL3].index(a), [canon(m) for m in AAL3].index(b)
    res["connections"].append(dict(conn=f"{a}--{b}", ref=share(ref_sel, i90, j90), yeh_at_90=share(yeh, i90, j90),
                                   label_at_90_under_AAL3=f"{AAL3[i90]}--{AAL3[j90]}", yeh_at_3=share(yeh, i3, j3)))
# strongest mean edges of the AAL3-candidate cohorts under each labelling
m = W[yeh].mean(0); iu90 = np.triu_indices(90, 1); o = np.argsort(m[iu90])[::-1][:5]
res["top_edges"] = [dict(share=float(m[iu90][e] * 1000), as_AAL90=f"{AAL90[iu90[0][e]]}--{AAL90[iu90[1][e]]}",
                         as_AAL3=f"{AAL3[iu90[0][e]]}--{AAL3[iu90[1][e]]}") for e in o]
json.dump(res, open(OUT + "atlas_check.json", "w"), indent=1)
np.savez(OUT + "heatmaps.npz", ref=refW, hcpd90=W[ds == 6].mean(0)[np.ix_(I90, I90)], hcpd3=W[ds == 6].mean(0)[np.ix_(I3, I3)],
         shared=np.array(SHARED))
res["cohort"]

## Figure (SI)

In [ ]:
import json
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

D = OUT; FIGS = REPO / "paper/figures/p1"
res = json.load(open(D + "atlas_check.json")); H = np.load(D + "heatmaps.npz")
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
C90, C3 = "#2a78d6", "#eb6834"                      # categorical slots 1-2 (validated)
SEQ = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
cmap = LinearSegmentedColormap.from_list("seq", SEQ)
mpl.rcParams.update({"font.family": "DejaVu Sans", "font.size": 7, "axes.edgecolor": INK2, "axes.labelcolor": INK,
                     "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "axes.linewidth": 0.6,
                     "xtick.major.width": 0.6, "ytick.major.width": 0.6, "pdf.fonttype": 42})

fig = plt.figure(figsize=(6.9, 2.55))
gs = fig.add_gridspec(1, 5, width_ratios=[1.25, 0.30, 1, 1, 1], wspace=0.07, left=0.165, right=0.925, top=0.80, bottom=0.20)

# (a) dumbbell: agreement with the reference under each labelling
ax = fig.add_subplot(gs[0, 0])
order = ["dHCP", "BCP", "HCPd", "HCPya", "camCAN", "HCPa"]
y = np.arange(len(order))[::-1]
for yi, c in zip(y, order):
    v = res["cohort"][c]
    ax.plot([v["rP_90"], v["rP_3"]], [yi, yi], color=GRID, lw=1.5, zorder=1, solid_capstyle="round")
    ax.scatter(v["rP_90"], yi, s=34, color=C90, edgecolor="#fcfcfb", linewidth=1.0, zorder=3)
    ax.scatter(v["rP_3"], yi, s=34, color=C3, edgecolor="#fcfcfb", linewidth=1.0, zorder=3)
ax.set_yticks(y); ax.set_yticklabels([f"{c} (n={res['cohort'][c]['n']})" for c in order])
ax.set_xlim(0, 1); ax.set_xticks([0, 0.25, 0.5, 0.75, 1]); ax.set_ylim(-0.6, len(order) - 0.4)
ax.xaxis.grid(True, color=GRID, lw=0.5); ax.set_axisbelow(True)
for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.set_xlabel("correlation with the reference\n(edge presence, 82 shared regions)", color=INK2, fontsize=6.5)
ax.axhline(3.5, color=INK2, lw=0.5, ls=(0, (2, 2)))
ax.text(-0.52, 1.13, "a", fontsize=9, fontweight="bold", transform=ax.transAxes)
h1 = ax.scatter([], [], s=34, color=C90, label="read with AAL90 labels")
h2 = ax.scatter([], [], s=34, color=C3, label="read with AAL3 labels")
ax.legend(handles=[h1, h2], loc="lower left", bbox_to_anchor=(-0.04, 1.0), ncol=1, frameon=False, handletextpad=0.2, labelspacing=0.3)

# (b) mean connectivity on the 82 shared regions, AAL90 order
sh = list(H["shared"]); lr = [i for i, n in enumerate(sh) if n.endswith("_L")] + [i for i, n in enumerate(sh) if n.endswith("_R")]
mats = [H[k][np.ix_(lr, lr)] for k in ("ref", "hcpd90", "hcpd3")]
titles = ["Reference:\nCALM/RED/ACE", "HCPd,\nread with AAL90 labels", "HCPd,\nread with AAL3 labels"]
vals = np.concatenate([m[m > 0] for m in mats]) * 1000
lo, hi = np.log10(np.percentile(vals, 5)), np.log10(np.percentile(vals, 99.5))
for k, (m, t) in enumerate(zip(mats, titles)):
    a = fig.add_subplot(gs[0, 2 + k])
    z = np.where(m > 0, np.log10(np.clip(m * 1000, 10 ** lo, None)), np.nan)
    im = a.imshow(z, cmap=cmap, vmin=lo, vmax=hi, interpolation="nearest")
    a.set_facecolor("#fcfcfb"); a.set_xticks([]); a.set_yticks([])
    for s in a.spines.values(): s.set_color(GRID); s.set_linewidth(0.5)
    a.set_title(t, fontsize=6.5, color=INK, pad=3)
    a.axhline(40.5, color=INK2, lw=0.4); a.axvline(40.5, color=INK2, lw=0.4)
    if k == 0:
        a.text(-0.12, 1.13, "b", fontsize=9, fontweight="bold", transform=a.transAxes)
        a.set_ylabel("right        left", fontsize=6, color=INK2, labelpad=2)
    a.set_xlabel("left        right", fontsize=6, color=INK2, labelpad=2)
cax = fig.add_axes([0.935, 0.20, 0.009, 0.60])
cb = fig.colorbar(im, cax=cax); cb.outline.set_visible(False); cb.ax.tick_params(labelsize=6, length=2)
ticks = [t for t in [-2, -1, 0, 1] if lo <= t <= hi]; cb.set_ticks(ticks); cb.set_ticklabels([f"$10^{{{t}}}$" for t in ticks])
cb.set_label("share of total weight (‰)", fontsize=6, color=INK2, labelpad=1)
fig.text(0.70, 0.045, "Mean connectivity (log scale) on the 82 regions shared by AAL90 and AAL3,\nleft-hemisphere regions first, each block in AAL90 order. Blank: no edge in any subject.", fontsize=6, color=INK2, ha="center")
fig.savefig(FIGS / "figS_atlas.pdf"); fig.savefig(FIGS / "figS_atlas.png", dpi=600)
print("saved")